# Phase Trombone Map Demo

This notebook demonstrates the map used by `trombone_utils.jl`.

The key idea is

```text
M_trombone = A * R(delta_phi) * inv(A)
```

where `A` is the local Courant-Snyder normalization matrix built from Twiss `beta` and `alpha`, and `R` is a phase rotation in normalized coordinates.

The current Chapter 12 utility uses this uncoupled construction separately in the two transverse planes, then adds dispersion and path-length terms so the 6D first-order map remains symplectic.

In [ ]:
using LinearAlgebra
using Printf

## 1. One transverse plane

For one uncoupled transverse plane, the local Twiss matrix is

```text
A = [ sqrt(beta)        0
     -alpha/sqrt(beta)  1/sqrt(beta) ]
```

A pure phase trombone rotates in normalized coordinates by `dphi`.

In [ ]:
beta = 23.0
alpha = -1.4
dphi = 0.37

A = [sqrt(beta) 0.0;
     -alpha / sqrt(beta) 1 / sqrt(beta)]

R = [cos(dphi) sin(dphi);
     -sin(dphi) cos(dphi)]

M_from_A = A * R * inv(A)

gamma = (1 + alpha^2) / beta
M_expanded = [cos(dphi) + alpha * sin(dphi)  beta * sin(dphi);
              -gamma * sin(dphi)             cos(dphi) - alpha * sin(dphi)]

@printf("max |A R A^-1 - expanded| = %.3e\n", maximum(abs.(M_from_A .- M_expanded)))
M_from_A

The expanded matrix is exactly the formula used in `trombone_utils.jl` for each plane.

## 2. Build the current 6D trombone matrix

The Chapter 12 map uses coordinates

```text
(x, px, y, py, z, pz)
```

It builds two transverse phase rotations, adds dispersion terms proportional to `pz`, and adds the matching `z` terms required for first-order symplecticity.

In [ ]:
function trombone_matrix_6d(dnu_x, dnu_y, beta_x, alpha_x, beta_y, alpha_y,
                            eta_x, etap_x, eta_y, etap_y)
    A11 = cos(dnu_x) + alpha_x * sin(dnu_x)
    A12 = beta_x * sin(dnu_x)
    A21 = -(1 + alpha_x^2) / beta_x * sin(dnu_x)
    A22 = cos(dnu_x) - alpha_x * sin(dnu_x)

    Dx = (1 - A11) * eta_x - A12 * etap_x
    Dpx = -A21 * eta_x + (1 - A22) * etap_x

    Tx = -A11 * Dpx + A21 * Dx
    Tpx = -A12 * Dpx + A22 * Dx

    B11 = cos(dnu_y) + alpha_y * sin(dnu_y)
    B12 = beta_y * sin(dnu_y)
    B21 = -(1 + alpha_y^2) / beta_y * sin(dnu_y)
    B22 = cos(dnu_y) - alpha_y * sin(dnu_y)

    Dy = (1 - B11) * eta_y - B12 * etap_y
    Dpy = -B21 * eta_y + (1 - B22) * etap_y

    Ty = -B11 * Dpy + B21 * Dy
    Tpy = -B12 * Dpy + B22 * Dy

    return [
        A11  A12  0.0  0.0  0.0  Dx;
        A21  A22  0.0  0.0  0.0  Dpx;
        0.0  0.0  B11  B12  0.0  Dy;
        0.0  0.0  B21  B22  0.0  Dpy;
        Tx   Tpx  Ty   Tpy  1.0  0.0;
        0.0  0.0  0.0  0.0  0.0  1.0
    ]
end

In [ ]:
dnu_x = 0.42
dnu_y = -0.31

beta_x, alpha_x = 18.0, 0.8
beta_y, alpha_y = 31.0, -1.1

eta_x, etap_x = 0.55, 0.021
eta_y, etap_y = 0.04, -0.006

M6 = trombone_matrix_6d(dnu_x, dnu_y, beta_x, alpha_x, beta_y, alpha_y,
                        eta_x, etap_x, eta_y, etap_y)

round.(M6; digits=6)

## 3. Symplectic check

For canonical coordinates `(x, px, y, py, z, pz)`, the symplectic condition is

```text
transpose(M) * J * M = J
```

In [ ]:
function canonical_J(n_pairs)
    J = zeros(2n_pairs, 2n_pairs)
    for i in 1:n_pairs
        a = 2i - 1
        b = 2i
        J[a, b] = 1
        J[b, a] = -1
    end
    return J
end

J6 = canonical_J(3)
symplectic_error = norm(transpose(M6) * J6 * M6 - J6)
@printf("||M' J M - J|| = %.3e\n", symplectic_error)

The two transverse eigenvalue pairs should lie on the unit circle, with phases close to `+-dnu_x` and `+-dnu_y`. The longitudinal pair is `1, 1` for this simple trombone map.

In [ ]:
vals = eigvals(M6)
for lambda in vals
    @printf("lambda = % .6f %+ .6fi, |lambda| = %.6f, angle = % .6f rad\n",
            real(lambda), imag(lambda), abs(lambda), angle(lambda))
end

## 4. Why the dispersion terms are there

If a particle is on the local dispersive closed orbit

```text
x  = eta_x  * pz
px = etap_x * pz
y  = eta_y  * pz
py = etap_y * pz
```

then the trombone should not kick it away from that dispersive orbit. The `Dx`, `Dpx`, `Dy`, and `Dpy` terms enforce that.

In [ ]:
pz = 0.003
v_disp = [eta_x * pz, etap_x * pz, eta_y * pz, etap_y * pz, 0.0, pz]
v_out = M6 * v_disp

@printf("input transverse  = %s\n", string(round.(v_disp[1:4]; digits=10)))
@printf("output transverse = %s\n", string(round.(v_out[1:4]; digits=10)))
@printf("max transverse difference = %.3e\n", maximum(abs.(v_out[1:4] .- v_disp[1:4])))

## 5. Connection to `trombone_utils.jl`

The utility function does this same construction at a selected marker:

```julia
attach_trombone!(element, ring, twiss_table, dnu_x, dnu_y)
```

Internally it stores

```text
(dnu_x, dnu_y, beta_x, alpha_x, beta_y, alpha_y, eta_x, etap_x, eta_y, etap_y)
```

as `transport_map_params`. During tracking, SciBmad calls the custom `transport_map` with those parameters.

The original accumulated phase `phi_1` or `phi_2` is not needed for the trombone map itself. Only the local normalization `A` and the requested additional phase advances are needed.